# MediPredict AI — Exploratory Data Analysis & Point-in-Time Audit

**Project:** Hospital Appointment No-Show Prediction & Operations Intelligence  
**Organization:** ALGIONS AI LABS PRIVATE LIMITED  
**Author:** AI/ML Engineering  

### Objectives:
1. Ingest and profile the appointment dataset.
2. Conduct rigorous data hygiene checks (out-of-range ages, negative waiting days, schema validation).
3. Investigate the relationship between booking-time features and appointment attendance.
4. Audit point-in-time features to prevent data leakage and enforce non-clinical governance boundaries.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# Path to data
data_path = Path("../data/raw/appointments.csv")
df_raw = pd.read_csv(data_path)
print(f"Raw dataset loaded: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns")
df_raw.head(3)

## 1. Schema Profiling & Data Quality Audit

In [2]:
# Check missing values
null_counts = df_raw.isnull().sum()
print("Missing value summary:")
print(null_counts[null_counts > 0] if null_counts.sum() > 0 else "No missing values detected!")

# Check Age bounds
invalid_ages = df_raw[(df_raw["Age"] < 0) | (df_raw["Age"] > 115)]
print(f"Invalid age records: {len(invalid_ages)}")

# Parse dates and compute waiting days
sched_dt = pd.to_datetime(df_raw["ScheduledDay"], utc=True)
app_dt = pd.to_datetime(df_raw["AppointmentDay"], utc=True)
waiting_days = (app_dt.dt.floor("D") - sched_dt.dt.floor("D")).dt.days

negative_wait = (waiting_days < 0).sum()
print(f"Negative waiting days records (temporal causality violations): {negative_wait}")

## 2. Target Variable Distribution (`No-show`)

In [3]:
target_counts = df_raw["No-show"].value_counts()
target_pct = df_raw["No-show"].value_counts(normalize=True) * 100

target_summary = pd.DataFrame({"Count": target_counts, "Percentage (%)": target_pct.round(2)})
print(target_summary)

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=target_counts.index, y=target_counts.values, palette=["#22c55e", "#f87171"], ax=ax)
ax.set_title("Distribution of Target Variable (No-Show)")
ax.set_ylabel("Number of Appointments")
plt.show()

## 3. Lead Time (Waiting Days) vs Missed Appointment Rate

Hypothesis: Appointments scheduled far in advance experience higher cancellation / missed rates compared to immediate or same-day bookings.

In [4]:
# Clean sample for analysis
df_clean = df_raw[(df_raw["Age"] >= 0) & (df_raw["Age"] <= 115)].copy()
df_clean["waiting_days"] = waiting_days
df_clean = df_clean[df_clean["waiting_days"] >= 0]
df_clean["no_show_binary"] = (df_clean["No-show"] == "Yes").astype(int)

# Bin waiting days
df_clean["wait_bucket"] = pd.cut(
    df_clean["waiting_days"],
    bins=[-1, 0, 3, 7, 14, 30, 90],
    labels=["0 (Same-day)", "1-3 days", "4-7 days", "8-14 days", "15-30 days", "30+ days"]
)

wait_agg = df_clean.groupby("wait_bucket", observed=False)["no_show_binary"].agg(["count", "mean"]).reset_index()
wait_agg["no_show_rate_pct"] = wait_agg["mean"] * 100

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=wait_agg, x="wait_bucket", y="no_show_rate_pct", palette="teal", ax=ax)
ax.set_title("No-Show Rate by Lead Time (Waiting Days)")
ax.set_ylabel("No-Show Rate (%)")
ax.set_xlabel("Booking Lead Time")
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points')
plt.show()

## 4. Age Demographic Patterns

In [5]:
df_clean["age_group"] = pd.cut(
    df_clean["Age"],
    bins=[-1, 12, 24, 39, 59, 120],
    labels=["Pediatric (0-12)", "Youth (13-24)", "Adult (25-39)", "Middle Age (40-59)", "Senior (60+)"]
)
age_agg = df_clean.groupby("age_group", observed=False)["no_show_binary"].agg(["count", "mean"]).reset_index()
age_agg["no_show_rate_pct"] = age_agg["mean"] * 100

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=age_agg, x="age_group", y="no_show_rate_pct", palette="Blues_d", ax=ax)
ax.set_title("No-Show Rate by Age Cohort")
ax.set_ylabel("No-Show Rate (%)")
plt.show()

## 5. Point-in-Time Leakage & Feature Governance Audit

### Prohibited Column Check
We verify that the proposed model feature matrix excludes all direct identifiers, sensitive clinical variables, and post-booking events:

In [6]:
from src.config import FEATURE_POLICY, ALL_MODEL_FEATURES

print("Model Features Approved for Training:")
for feat in ALL_MODEL_FEATURES:
    print(f"  [INCLUDED]  {feat}")

print("\nFeatures Excluded by Governance & Leakage Policy:")
for col, meta in FEATURE_POLICY.items():
    if meta["status"] == "EXCLUDED":
        print(f"  [EXCLUDED]  {col:<15} ({meta['category']}) -> {meta['reason']}")

## 6. Summary of Key Insights for Model Development

1. **Imbalance:** Positive class (`No-show = Yes`) is ~20% of all scheduled appointments. Baseline accuracy is misleading; evaluation must center on **Precision, Recall, F1, PR-AUC, and ROC-AUC**.
2. **Lead Time is the Single Strongest Predictor:** Same-day bookings have a <5% no-show rate, while appointments scheduled 14+ days in advance have >30% no-show rate.
3. **Point-in-Time Discipline:** `SMS_received` is recorded days after booking and must be excluded from scheduling-time models.
4. **Governance:** Clinical diagnoses are excluded. The system serves as an operational prioritization tool, not a clinical triage system.